In [1]:
import cv2
import numpy as np
from ultralytics import YOLO
import torch
import os
from collections import deque
from catboost import CatBoostRegressor

In [22]:
cnt = 0
cap = cv2.VideoCapture("VID_20260411_212904.mp4")
while cap.isOpened():
    ret , frame = cap.read()
    if not ret:
        break
    if cnt % 10 == 9: # можно не 10 но я захотел 10
        cv2.imwrite(f"images/{cnt}.jpg", frame)
    cnt += 1
cap.release()

In [ ]:
input_dir = 'images' # все по brg как требует opencv
output_dir = 'labels'
LOWER_BLUE = np.array([80, 150, 50])  
UPPER_BLUE = np.array([100, 255, 255]) 
for i in os.listdir(input_dir):
    img = cv2.imread(os.path.join(input_dir, i)) # склейка пути
    img = cv2.GaussianBlur(img, (7, 7), 0) 
    h, w, _ = img.shape # _ это заглушка 
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV) 
    mask = cv2.inRange(hsv, LOWER_BLUE, UPPER_BLUE) 
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE) # ищет белый цвет
    coords = [] # для хранения координат как требует yolo
    for cnt in contours:    
        bx, by, bw, bh = cv2.boundingRect(cnt) # описывает контур в квадрат
        if cv2.contourArea(cnt) < 20: # убираем мелкий шум
            continue
        coords.append(f"0 {(bx+bw/2)/w:.6f} {(by+bh/2)/h} {bw/w} {bh/h}") # так требует yolo
    with open(os.path.join(output_dir, os.path.splitext(i)[0] + ".txt"), "w") as f: # генерит имя файла и делает доступным для записи
        f.write("\n".join(coords)) # сохраняет найденные координаты

In [ ]:
model = YOLO('yolo26m.pt')
if __name__ == '__main__':
    model.train(
        data = r'C:\Users\user\projectolymp\data.yaml',
        epochs=80,
        imgsz=640, 
        batch=12,
        device="cuda" if torch.cuda.is_available() else "cpu", 
        project="models",
        name="1",
        hsv_h=0.015,     # аугментации
        hsv_s=0.7, 
        hsv_v=0.4,    
        degrees=15.0,
        translate=0.1,  
        scale=0.54,   
        shear=0.0,      
        flipud=0.0, 
        fliplr=0.54,     
        mosaic=1.0,  
        mixup=0.1
    )

In [ ]:
model = YOLO(r'C:\Users\user\projectolymp\runs\detect\models\110\weights\best.pt')
cb_model = CatBoostRegressor()
cb_model.load_model("catboost.cbm")
cap = cv2.VideoCapture(0)
track_history = {}
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: 
        break
    results = model.track(frame, persist=True) # ищет маркеры и присваевает им id
    if results[0].boxes.id is not None: # провеййрка нашла ли модель хоть 1 маркер
        boxes = results[0].boxes.xywh.cpu().numpy() # получение координатов объектов
        track_ids = results[0].boxes.id.int().cpu().tolist() # получение id
        for box, track_id in zip(boxes, track_ids): # проходим по координате и ее id
            x, y, w, h = box # распаковка
            track = track_history.get(track_id, deque(maxlen=150)) # история 5 секунд при 30 фпс
            track.append((float(x), float(y))) # добавляем точку в историю движения 
            track_history[track_id] = track # сохраняем обновленную историю
            if len(track) >= 10:
                last_10_points = list(track)[-10:]
                input_data = [coord for pt in last_10_points for coord in pt]
                prediction = cb_model.predict([input_data]) # вернет predicted_x и predicted_y
                cv2.circle(frame, (int(prediction[0][0]), int(prediction[0][1])), 7, (255, 0, 0), -1) # отрисовка точки
            for i in range(1, len(track)): # отрисовка
                pt1 = (int(track[i-1][0]), int(track[i-1][1]))
                pt2 = (int(track[i][0]), int(track[i][1]))
                cv2.line(frame, pt1, pt2, (0, 0, 255), 2) # красная линия толщиной 2 пикселя
    cv2.imshow("MMC", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"): # на q закрытие
        break
cap.release()
cv2.destroyAllWindows()